In [68]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv026931/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [69]:
from pyspark.sql.functions import *

In [70]:
loans_schema = '''loan_id string,member_id string,loan_amount
float,funded_amount float,loan_term_months string,interest_rate
float,monthly_installment float,issue_date string,loan_status
string,loan_purpose string,loan_title string'''

In [71]:
loans_raw_df = spark.read \
.format("csv") \
.schema(loans_schema) \
.option("header",True) \
.load("/user/itv026931/lendingclubproject/raw/loans_data_csv")

In [10]:
loans_raw_df

loan_id,member_id,loan_amount,funded_amount,loan_term_months,interest_rate,monthly_installment,issue_date,loan_status,loan_purpose,loan_title
6300587,e9f690be7e25fc285...,12000.0,12000.0,36 months,15.22,417.28,Jul-2013,Fully Paid,credit_card,Credit Card Payoff
6311418,7602c853409dfc8d8...,7125.0,7125.0,36 months,15.88,250.08,Jul-2013,Fully Paid,other,Other
6321535,5e1091070269a95ad...,24000.0,24000.0,60 months,14.33,562.56,Aug-2013,Fully Paid,credit_card,Loan for life
6300526,cace2e77b60e12156...,24000.0,24000.0,36 months,6.03,730.46,Jul-2013,Fully Paid,credit_card,Refinance loan
6301469,54f7c754fbe1cdad5...,18000.0,18000.0,60 months,17.56,452.79,Aug-2013,Fully Paid,debt_consolidation,Debt consolidation
6306881,1e13cab7268220e9e...,20000.0,20000.0,60 months,11.55,440.36,Aug-2013,Fully Paid,debt_consolidation,Debt Consolidation
6301579,6c8dfadce31c466b6...,12000.0,12000.0,36 months,11.55,396.0,Jul-2013,Fully Paid,debt_consolidation,Goodbye Stress
6291661,2b0dcdeaf82ddc95c...,28000.0,28000.0,60 months,16.78,692.57,Aug-2013,Fully Paid,debt_consolidation,Debt consolidation
6311612,882d6a9a903c44fc2...,5500.0,5500.0,36 months,10.64,179.13,Aug-2013,Charged Off,credit_card,Yes I Can Manage ...
6311704,946424debb78b996f...,23600.0,23600.0,60 months,12.35,529.16,Jul-2013,Fully Paid,debt_consolidation,Consolidation


ADDING A COLUMN WITH CURRENT TIME STAMP

In [72]:
loans_raw_ingestd = loans_raw_df.withColumn("ingest_date",current_timestamp())

In [73]:
loans_raw_ingestd.createOrReplaceTempView("loans")

In [74]:
loans_raw_ingestd.count()

2260701

Remove all the rows having NULL

In [75]:
columns_to_check = ["loan_amount", "funded_amount", "loan_term_months","interest_rate","monthly_installment",
                    "issue_date","loan_status","loan_purpose"]

In [76]:
loans_filtered_df = loans_raw_ingestd.na.drop(subset=columns_to_check)

In [20]:
loans_filtered_df.count()

2260667

In [77]:
loans_filtered_df.createOrReplaceTempView("loans")

MODIFYING A COLUMN HAVING "36 months" TO '3' YEARS

In [78]:
loans_term_modified_df = loans_filtered_df.withColumn("loan_term_months",(regexp_replace(col("loan_term_months")," months","").cast("int") / 12).cast("int")) \
.withColumnRenamed("loan_term_months","loan_term_years")

In [79]:
loans_term_modified_df_new = loans_filtered_df.withColumn("loan_term_months",(regexp_replace("loan_term_months","\D","").cast("int") / 12).cast("int")) \
.withColumnRenamed("loan_term_months","loan_term_years")

In [44]:
loans_term_modified_df_new.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amount: float (nullable = true)
 |-- funded_amount: float (nullable = true)
 |-- loan_term_years: integer (nullable = true)
 |-- interest_rate: float (nullable = true)
 |-- monthly_installment: float (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- loan_title: string (nullable = true)
 |-- ingest_date: timestamp (nullable = false)



In [47]:
loans_term_modified_df.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amount: float (nullable = true)
 |-- funded_amount: float (nullable = true)
 |-- loan_term_years: integer (nullable = true)
 |-- interest_rate: float (nullable = true)
 |-- monthly_installment: float (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- loan_title: string (nullable = true)
 |-- ingest_date: timestamp (nullable = false)



In [80]:
loans_term_modified_df.createOrReplaceTempView("loans")

In [94]:
loan_purpose_modified.groupBy("loan_purpose").agg(count("*").alias("total")).orderBy(col("total").desc())

loan_purpose,total
debt_consolidation,1277790
credit_card,516926
home_improvement,150440
other,139667
major_purchase,50429
medical,27481
small_business,24659
car,24009
vacation,15525
moving,15402


In [90]:
spark.sql("select loan_purpose, count(*) as count from loans group by loan_purpose order by count desc")

loan_purpose,count
debt_consolidation,1277790
credit_card,516926
home_improvement,150440
other,139667
major_purchase,50429
medical,27481
small_business,24659
car,24009
vacation,15525
moving,15402


REMOVING UNWANTED LOAN PURPOSE AND REPLACING THAT WITH "Others"

In [82]:
loan_purpose_lookup = ["debt_consolidation", "credit_card","home_improvement", "other", "major_purchase", "medical", "small_business",
"car", "vacation", "moving", "house", "wedding", "renewable_energy","educational"]

In [88]:
loan_purpose_modified = loans_term_modified_df.withColumn("loan_purpose",when(col("loan_purpose").isin(loan_purpose_lookup),col("loan_purpose")).otherwise("other"))

In [95]:
loan_purpose_modified.createOrReplaceTempView("loans")

In [99]:
loan_purpose_modified.write \
.option("header",True) \
.format("parquet") \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/cleaned/loans_parquet") \
.save()

In [100]:
loan_purpose_modified.write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/cleaned/loans_csv") \
.save()